# AmsiNet - Step 3a: hierarchy data and the four agents, trained separately

**Runtime:** Colab A100. Needs `MyDrive/amsinet/data/bstd_index.json`, `crops_meta.csv`, `crops224.tar` (Steps 1-2) and the five flat-baseline result files in `MyDrive/amsinet/results/`.

What this notebook does
1. Cuts page, block and line regions (with soft script-distribution labels) from the BSTD images once. This needs the original images, so the first run downloads `detection.zip` again (about 5-10 minutes) unless the region crops are already saved on Drive.
2. Derives the expert groups for the word agent by clustering the average **validation** confusion matrix of the flat baseline (the test set is never used for design).
3. Trains the page, block, line and word agents separately (word agent with and without the mixture of experts), and reports for each level (a) region-level accuracy and (b) word-level accuracy if its prediction is broadcast to the words it contains.
4. Computes the oracle ceilings (a perfect agent at each level) and measures params / FLOPs / CPU latency at the real input sizes.

Design notes to carry into Section 3 of the paper
- The line agent uses 64x512 inputs with a patch-16 ViT-Small (the manuscript currently says 32x256 with 4x4 patches; Section 3.3 will be changed to match what is actually run).
- The word-agent gate is supervised by the expert-group label (an auxiliary loss), which replaces the pure load-balancing loss; `LAMBDA_BAL` is kept as an option and defaults to 0.
- Hierarchy levels are geometry-derived (not human-annotated); state this in the paper.

In [ ]:
# 1. Setup
from google.colab import drive
import os
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
assert os.path.exists('/content/drive/MyDrive/amsinet/data/bstd_index.json'), 'Drive is not mounted or amsinet/data is missing: delete the runtime, reconnect, and run this cell first'
!pip -q install timm gdown
import torch, timm
print('torch', torch.__version__, '| timm', timm.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (switch runtime to GPU)')

In [ ]:
# 2. Configuration, index, and restore of saved crops
import json, os, glob, math, random, time, subprocess
import numpy as np, pandas as pd
from collections import Counter

ROOT, LOCAL = '/content/drive/MyDrive/amsinet', '/content/bstd'
os.makedirs(f'{ROOT}/agents', exist_ok=True); os.makedirs(LOCAL, exist_ok=True)

CFG = dict(SEEDS=[0, 1, 2, 3, 4], LEVELS=('block', 'line', 'word', 'word_moe'), EPOCHS=8, BATCH=64, LR=1e-4, WD=0.05, WORKERS=8, PAD=0.04,
           LINE_HW=(64, 512), M_GROUPS=4, LAMBDA_GATE=0.3, LAMBDA_BAL=0.0, SMOOTH_WORD=0.1)

D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8'))
IMG_ROOT, SCRIPTS, INDEX = D['img_root'], D['scripts'], D['index']
SIDX = {s: i for i, s in enumerate(SCRIPTS)}; C = len(SCRIPTS)

CROPS, CROPS_TAR = f'{LOCAL}/crops224', f'{ROOT}/data/crops224.tar'
REG, REG_TAR = f'{LOCAL}/reg', f'{ROOT}/data/reg_crops.tar'
for d, tar in ((CROPS, CROPS_TAR), (REG, REG_TAR)):
    if not os.path.isdir(d) and os.path.exists(tar):
        print('restoring', tar); subprocess.run(['tar', '-xf', tar, '-C', LOCAL], check=True)
META = pd.read_csv(f'{ROOT}/data/crops_meta.csv')            # word crops from Step 2
print(len(INDEX), 'images |', C, 'scripts | word crops:', len(META), '| word crops on disk:', os.path.isdir(CROPS), '| region crops on disk:', os.path.isdir(REG))

## 3. Region geometry, soft labels and the word-label table (pure Python, no GPU)

In [ ]:
def valid(w): return (w['bbox'][2] - w['bbox'][0]) >= 4 and (w['bbox'][3] - w['bbox'][1]) >= 4

def union(boxes):
    b = np.array(boxes, float); return [b[:, 0].min(), b[:, 1].min(), b[:, 2].max(), b[:, 3].max()]

def soft(ids, W):
    v = np.zeros(C)
    for i in ids:
        if W[i]['script']: v[SIDX[W[i]['script']]] += 1
    return (v / v.sum()).tolist() if v.sum() > 0 else None

def regions_of(rec):
    # page = whole image; block / line = union of the boxes of the words in the group.
    # Label = distribution of scripts over the *labelled* words inside (Section 3.1).
    W = rec['words']; ok = [i for i, w in enumerate(W) if valid(w)]
    lab = [i for i in ok if W[i]['script']]
    regs = []
    if lab: regs.append(dict(level='page', rid='p', words=lab, box=None, soft=soft(lab, W)))
    for key in ('block', 'line'):
        groups = {}
        for i in ok: groups.setdefault(W[i].get(key, -1), []).append(i)
        for gid, ids in groups.items():
            l = [i for i in ids if W[i]['script']]
            if l: regs.append(dict(level=key, rid=f'{key[0]}{gid}', words=l, box=union([W[i]['bbox'] for i in ids]), soft=soft(l, W)))
    return regs

# true script of every labelled word, keyed like the Step-2 word crops
WLAB = {(r['id'], k): SIDX[w['script']] for r in INDEX for k, w in enumerate(r['words']) if w['script'] and valid(w)}
print('labelled words:', len(WLAB))

## 4. Cut the region crops (one-time; resumable). Needs the original images, so it downloads them if they are not on the disk.

In [ ]:
from PIL import Image, ImageOps
from concurrent.futures import ThreadPoolExecutor
Image.MAX_IMAGE_PIXELS = None
FILL = (124, 116, 104); LH, LW = CFG['LINE_HW']

def letterbox(im, w, h):
    iw, ih = im.size; s = min(w / iw, h / ih)
    nw, nh = max(1, int(round(iw * s))), max(1, int(round(ih * s)))
    im = im.resize((nw, nh), Image.BICUBIC, reducing_gap=3.0)
    cv = Image.new('RGB', (w, h), FILL); cv.paste(im, ((w - nw) // 2, (h - nh) // 2)); return cv

def padbox(b, im, pad):
    x1, y1, x2, y2 = b; px, py = (x2 - x1) * pad, (y2 - y1) * pad
    return (max(0, int(x1 - px)), max(0, int(y1 - py)), min(im.width, int(math.ceil(x2 + px))), min(im.height, int(math.ceil(y2 + py))))

SKIPPED = []
def extract_rec(rec):
    rows, im = [], None
    for r in regions_of(rec):
        d = f"{REG}/{r['level']}/{rec['split']}"; p = f"{d}/{rec['id']}_{r['rid']}.jpg"
        if not os.path.exists(p):
            if im is None:
                im = ImageOps.exif_transpose(Image.open(os.path.join(IMG_ROOT, rec['image']))).convert('RGB')
            os.makedirs(d, exist_ok=True)
            if r['level'] == 'page':
                crop = letterbox(im, 224, 224)
            else:
                bx = padbox(r['box'], im, CFG['PAD'])
                if bx[2] - bx[0] < 2 or bx[3] - bx[1] < 2:      # box outside the image or degenerate
                    SKIPPED.append((rec['id'], r['rid'], r['level'])); continue
                crop = im.crop(bx)
                if min(crop.size) < 2: continue
                crop = letterbox(crop, 224, 224) if r['level'] == 'block' else letterbox(crop, LW, LH)
            crop.save(p, quality=95)
        rows.append((r['level'], p, rec['split'], rec['id'], r['rid'], json.dumps(r['soft']), json.dumps(r['words'])))
    return rows

def ensure_images():
    global IMG_ROOT
    if os.path.isdir(IMG_ROOT): return
    z = f'{LOCAL}/detection.zip'
    zdrive = f'{ROOT}/data/detection.zip'          # permanent copy on Drive
    if not os.path.exists(z):
        if os.path.exists(zdrive):
            print('copying detection.zip from Drive (~5-10 min)...')
            subprocess.run(['cp', zdrive, z], check=True)
        else:
            import gdown
            gdown.download(id='1S7KUYfB-lQvbu6GtvZxxDPOD5ZZ080M0', output=z, quiet=False)
            if os.path.getsize(z) > 15e9:          # save a permanent copy to Drive (never hit the quota again)
                print('saving a copy to Drive...')
                subprocess.run(['cp', z, zdrive], check=True)
    subprocess.run(['unzip', '-q', '-o', z, '-d', LOCAL], check=True)
    IMG_ROOT = os.path.dirname(glob.glob(f'{LOCAL}/**/BSTD_v17.57.json', recursive=True)[0])

REG_CSV = f'{ROOT}/data/regions_meta.csv'
if os.path.exists(REG_CSV) and os.path.isdir(REG):
    REGDF = pd.read_csv(REG_CSV)
else:
    ensure_images()
    rows = []
    with ThreadPoolExecutor(6) as ex:
        for i, rr in enumerate(ex.map(extract_rec, INDEX)):
            rows.extend(rr)
            if (i + 1) % 500 == 0: print(f'{i + 1}/{len(INDEX)} images, {len(rows)} regions')
    REGDF = pd.DataFrame(rows, columns=['level', 'path', 'split', 'image_id', 'rid', 'soft', 'words'])
    REGDF.to_csv(REG_CSV, index=False)
    print('regions skipped (box outside image or degenerate):', len(SKIPPED))
    subprocess.run(['tar', '-cf', REG_TAR, '-C', LOCAL, 'reg'], check=True)
print(REGDF.groupby(['level', 'split']).size().unstack(1))

## 5. Oracle ceilings and the potential of context
For each level, the "oracle" assigns every word the **dominant true script of its region**. This is the best any agent at that level can do when its single prediction is broadcast to the words it contains. It shows how much finer levels can add over coarser ones.

In [ ]:
from sklearn.metrics import f1_score

def broadcast(df, cls):
    yt, yp = [], []
    for wj, c, iid in zip(df['words'], cls, df['image_id']):
        for wi in json.loads(wj): yt.append(WLAB[(iid, wi)]); yp.append(int(c))
    yt, yp = np.array(yt), np.array(yp); labs = list(range(C))
    return dict(acc=float((yt == yp).mean()), macro_f1=float(f1_score(yt, yp, average='macro', labels=labs, zero_division=0)),
                per_class_f1=f1_score(yt, yp, average=None, labels=labs, zero_division=0).tolist())

orc = []
for lvl in ('page', 'block', 'line'):
    df = REGDF[(REGDF.level == lvl) & (REGDF.split == 'test')]
    dom = [int(np.argmax(json.loads(s))) for s in df.soft]
    m = broadcast(df, dom); orc.append((lvl, len(df), round(100 * m['acc'], 2), round(100 * m['macro_f1'], 2)))
orc.append(('word', int((META.split == 'test').sum()), 100.0, 100.0))
ORACLE = pd.DataFrame(orc, columns=['level', 'test regions', 'oracle word acc (%)', 'oracle word macro-F1 (%)'])
display(ORACLE)

## 6. Expert groups from the 5-seed **validation** confusion matrix
Latin is kept as its own group (its errors are mostly a class-prior effect, not visual similarity). The other nine scripts are clustered by how often they are confused with each other. You may override `GROUP` by hand, but then say so in the paper.

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

files = sorted(glob.glob(f'{ROOT}/results/vit_base_patch16_224_seed*.json'))
Cm = sum(np.array(json.load(open(f))['val']['confusion'], float) for f in files)
Cn = Cm / Cm.sum(1, keepdims=True); S = (Cn + Cn.T) / 2; np.fill_diagonal(S, 0)
others = [i for i, s in enumerate(SCRIPTS) if s != 'Latin']
Sub = S[np.ix_(others, others)]
dist = 1 - Sub / Sub.max(); np.fill_diagonal(dist, 0)
lab = fcluster(linkage(squareform(dist, checks=False), 'average'), t=CFG['M_GROUPS'] - 1, criterion='maxclust')
GROUP = {'Latin': 0}; GROUP.update({SCRIPTS[i]: int(l) for i, l in zip(others, lab)})
# GROUP = {...}   # optional manual override
_g = f'{ROOT}/agents/word_moe_seed0.json'
if os.path.exists(_g): GROUP = json.load(open(_g))['groups']; print('using the expert groups saved with seed 0 (keeps all seeds comparable)')
M = max(GROUP.values()) + 1
for g in range(M): print('group', g, [s for s in SCRIPTS if GROUP[s] == g])
print('seeds used:', len(files), '| experts M =', M)

## 7. Models, data and the training loop

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.metrics import confusion_matrix
DEV = 'cuda'; GRP_T = torch.tensor([GROUP[s] for s in SCRIPTS])

class MoEHead(nn.Module):
    def __init__(s, d, ncls, M, hid=256):
        super().__init__(); s.gate = nn.Linear(d, M)
        s.experts = nn.ModuleList([nn.Sequential(nn.Linear(d, hid), nn.ReLU(), nn.Linear(hid, ncls)) for _ in range(M)])
    def forward(s, f):
        g = s.gate(f).softmax(-1); z = torch.stack([e(f) for e in s.experts], 1)
        return (g.unsqueeze(-1) * z).sum(1), g

class WordAgent(nn.Module):
    def __init__(s, M=None, name='efficientvit_b0.r224_in1k'):
        super().__init__(); s.bb = timm.create_model(name, pretrained=True, num_classes=0)
        s.bb.eval()
        with torch.no_grad(): d = s.bb(torch.zeros(2, 3, 224, 224)).shape[1]     # measured, not num_features (EfficientViT pools to 1280)
        s.bb.train()
        s.head = nn.Linear(d, C) if M is None else MoEHead(d, C, M)
    def forward(s, x):
        f = s.bb(x); o = s.head(f); return o if isinstance(o, tuple) else (o, None)

def build(level, pretrained=True):
    if level == 'page':  m = timm.create_model('mobilevit_xs.cvnets_in1k', pretrained=pretrained, num_classes=C)
    elif level == 'block': m = timm.create_model('convnext_tiny.fb_in22k', pretrained=pretrained, num_classes=C)
    elif level == 'line':  m = timm.create_model('vit_small_patch16_224.augreg_in21k', pretrained=pretrained, num_classes=C, img_size=(LH, LW))
    elif level == 'word':  return WordAgent(None)
    elif level == 'word_moe': return WordAgent(M)
    class W(nn.Module):
        def __init__(s, m): super().__init__(); s.m = m
        def forward(s, x): return s.m(x), None
    w = W(m); w.bb = m; return w

class DS(Dataset):
    def __init__(s, paths, soft, tf): s.p, s.t, s.tf = paths, soft.astype(np.float32), tf
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return s.tf(Image.open(s.p[i]).convert('RGB')), s.t[i]

def tables(level):
    if level.startswith('word'):
        out = {}
        for sp in ('train', 'val', 'test'):
            d = META[META.split == sp].reset_index(drop=True)
            y = np.eye(C, dtype=np.float32)[[SIDX[s] for s in d.script]]; out[sp] = (d, y)
        return out
    out = {}
    for sp in ('train', 'val', 'test'):
        d = REGDF[(REGDF.level == level) & (REGDF.split == sp)].reset_index(drop=True)
        out[sp] = (d, np.array([json.loads(x) for x in d.soft], dtype=np.float32))
    return out

def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

@torch.no_grad()
def predict(model, dl):
    model.eval(); P = []
    for x, _ in dl:
        with torch.autocast('cuda', dtype=torch.float16): lo, _ = model(x.to(DEV, non_blocking=True))
        P.append(lo.float().softmax(1).cpu())
    return torch.cat(P).numpy()

def dom_metrics(P, Y):
    pr, yt = P.argmax(1), Y.argmax(1); labs = list(range(C))
    return dict(acc=float((pr == yt).mean()), macro_f1=float(f1_score(yt, pr, average='macro', labels=labs, zero_division=0)))

def fit(level, seed):
    tag = f'{level}_seed{seed}'; out = f'{ROOT}/agents/{tag}.json'
    if os.path.exists(out): print('skip (done):', tag); return
    set_seed(seed); model = build(level).to(DEV); is_word = level.startswith('word')
    mc = timm.data.resolve_data_config({}, model=model.bb); norm = T.Normalize(mc['mean'], mc['std'])
    ttf = T.Compose([T.RandomAffine(5, translate=(0.03, 0.03), scale=(0.9, 1.1), fill=FILL), T.ColorJitter(0.3, 0.3, 0.2), T.ToTensor(), norm])
    etf = T.Compose([T.ToTensor(), norm]); TB = tables(level)
    dl = lambda sp, tf, sh: DataLoader(DS(TB[sp][0].path.tolist(), TB[sp][1], tf), batch_size=CFG['BATCH'], shuffle=sh, num_workers=CFG['WORKERS'], pin_memory=True, drop_last=sh)
    Ltr, Lva, Lte = dl('train', ttf, True), dl('val', etf, False), dl('test', etf, False)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['LR'], weight_decay=CFG['WD'])
    total = CFG['EPOCHS'] * len(Ltr); warm = max(1, int(0.05 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler('cuda'); best, best_state, hist = -1, None, []
    for ep in range(CFG['EPOCHS']):
        model.train(); t0 = time.time(); rl = 0
        for x, t in Ltr:
            x, t = x.to(DEV, non_blocking=True), t.to(DEV, non_blocking=True)
            if is_word: t = t * (1 - CFG['SMOOTH_WORD']) + CFG['SMOOTH_WORD'] / C
            with torch.autocast('cuda', dtype=torch.float16):
                lo, g = model(x)
                loss = -(t * lo.float().log_softmax(1)).sum(1).mean()
                if g is not None:
                    dom = t.argmax(1)
                    loss = loss + CFG['LAMBDA_GATE'] * F.nll_loss(torch.log(g.float() + 1e-8), GRP_T.to(DEV)[dom])
                    if CFG['LAMBDA_BAL'] > 0: loss = loss + CFG['LAMBDA_BAL'] * ((g.float().mean(0) - 1 / g.shape[1]) ** 2).sum()
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step(); rl += loss.item()
        mv = dom_metrics(predict(model, Lva), TB['val'][1])
        hist.append(dict(epoch=ep + 1, loss=rl / len(Ltr), **mv))
        print(f"{tag} ep{ep + 1}: loss {rl / len(Ltr):.4f} val acc {mv['acc']:.4f} macro-F1 {mv['macro_f1']:.4f} ({time.time() - t0:.0f}s)")
        if mv['macro_f1'] > best: best = mv['macro_f1']; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state); res = dict(level=level, seed=seed, history=hist, groups=GROUP, cfg=CFG)
    for sp, L in (('val', Lva), ('test', Lte)):
        P = predict(model, L); d = TB[sp][0]
        np.save(f'{ROOT}/agents/{tag}_{sp}_probs.npy', P.astype(np.float16)); d[['path']].to_csv(f'{ROOT}/agents/{tag}_{sp}_paths.csv', index=False)
        res[sp] = dict(region=dom_metrics(P, TB[sp][1]),
                       word=(broadcast(d, P.argmax(1)) if not is_word else {k: v for k, v in dom_metrics(P, TB[sp][1]).items()}))
    torch.save(best_state, f'{ROOT}/agents/{tag}.pt'); json.dump(res, open(out, 'w'))
    print(f"TEST {tag}: region acc {res['test']['region']['acc']:.4f} | word-level acc {res['test']['word']['acc']:.4f} macro-F1 {res['test']['word']['macro_f1']:.4f}")

## 8. Train the agents for the final five-seed results. Finished runs are skipped (seed 0 is already done). Each new seed takes roughly 15-20 minutes on an A100 (the page agent is not trained for seeds 1-4: it is not part of the final system, and its seed-0 result stays in the table as a reference).

In [ ]:
for seed in CFG['SEEDS']:
    for level in CFG['LEVELS']:
        fit(level, seed)

## 9. Results: per-level table (mean +/- 95% CI over seeds) next to the flat baseline and the oracle ceilings

In [ ]:
from scipy import stats
def mean_ci(v):
    v = np.asarray(v, float)
    return (float(v.mean()), float('nan')) if len(v) < 2 else (float(v.mean()), float(stats.t.ppf(0.975, len(v) - 1) * v.std(ddof=1) / math.sqrt(len(v))))
fmt = lambda v: (lambda m, c: f'{100 * m:.2f} +/- {100 * c:.2f}' if c == c else f'{100 * m:.2f}')(*mean_ci(v))
rows = []
for level in ('page', 'block', 'line', 'word', 'word_moe'):
    R = [json.load(open(f)) for f in sorted(glob.glob(f'{ROOT}/agents/{level}_seed*.json'))]
    if not R: continue
    rows.append(dict(agent=level, seeds=len(R), region_acc=fmt([r['test']['region']['acc'] for r in R]),
                     word_acc=fmt([r['test']['word']['acc'] for r in R]), word_macro_F1=fmt([r['test']['word']['macro_f1'] for r in R])))
fl = [json.load(open(f)) for f in sorted(glob.glob(f'{ROOT}/results/vit_base_patch16_224_seed*.json'))]
rows.append(dict(agent='flat ViT-B/16 (baseline)', seeds=len(fl), region_acc='-', word_acc=fmt([r['test']['acc'] for r in fl]), word_macro_F1=fmt([r['test']['macro_f1'] for r in fl])))
TAB = pd.DataFrame(rows); display(TAB); TAB.to_csv(f'{ROOT}/agents/agent_table.csv', index=False); display(ORACLE)

## 10. Parameters, FLOPs and CPU latency at the real input sizes (run after training; uses the CPU so it does not disturb anything)
FLOPs = 2 x multiply-accumulates, as before. CPU latency is the median of 30 single-image runs on this Colab CPU with all threads; state the CPU model in the paper.

In [ ]:
from torch.utils.flop_counter import FlopCounterMode
torch.set_num_threads(os.cpu_count())
def eff_cpu(level, shape, runs=30):
    m = build(level, pretrained=False).cpu().eval(); x = torch.randn(1, 3, *shape)
    with torch.no_grad(), FlopCounterMode(display=False) as fc: m(x)
    with torch.no_grad():
        for _ in range(5): m(x)
        ts = []
        for _ in range(runs): t = time.perf_counter(); m(x); ts.append((time.perf_counter() - t) * 1000)
    return round(sum(p.numel() for p in m.parameters()) / 1e6, 2), round(fc.get_total_flops() / 1e9, 2), round(float(np.median(ts)), 1)
sizes = {'page': (224, 224), 'block': (224, 224), 'line': (LH, LW), 'word': (224, 224), 'word_moe': (224, 224)}
EFF = pd.DataFrame([(k,) + eff_cpu(k, v) for k, v in sizes.items()], columns=['agent', 'params (M)', 'GFLOPs', 'CPU latency (ms)'])
display(EFF); EFF.to_csv(f'{ROOT}/agents/agent_efficiency_cpu.csv', index=False)
import platform; print('CPU threads:', os.cpu_count(), '|', platform.processor())

## 11. Look at the errors: crops of small-script words that the word agent called Latin

In [ ]:
import matplotlib.pyplot as plt
tag = 'word_moe_seed0'
P = np.load(f'{ROOT}/agents/{tag}_val_probs.npy').astype(np.float32); pa = pd.read_csv(f'{ROOT}/agents/{tag}_val_paths.csv')
val = META[META.split == 'val'].reset_index(drop=True); assert (val.path == pa.path).all()
truth = np.array([SIDX[s] for s in val.script]); pred = P.argmax(1)
bad = np.where((pred == SIDX['Latin']) & np.isin(truth, [SIDX[s] for s in ('Malayalam', 'Gujarati', 'Telugu', 'Odia')]))[0]
print(len(bad), 'val words of Malayalam/Gujarati/Telugu/Odia predicted as Latin')
fig, axs = plt.subplots(4, 6, figsize=(12, 8))
for ax, i in zip(axs.ravel(), np.random.RandomState(0).choice(bad, min(24, len(bad)), replace=False)):
    ax.imshow(Image.open(val.path[i])); ax.set_title(f'true {val.script[i][:8]}', fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()